# Idle-interactive: evidence, policy scenarios, and downside

This second notebook investigates the first opportunity using the same modules as the exporter. Run from top to bottom with **CutScope analytics**. All reclaim values are hypothetical sample resource-value scenarios. No session is terminated and no policy is deployed. Save executed copies in `generated/` or `analytics/notebooks/local/`.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PROJECT_SPEC.md").exists() and (p / "analytics").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open from the CutScope repository")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from analytics.load_data import load_official_data
from analytics.outcomes import build_summary
from analytics.pricing import Pricing
from analytics.scenarios import IdlePolicy, idle_hour_scenarios
from analytics.opportunities.idle_interactive import idle_candidates, build_idle_opportunity
from analytics.build_analysis import build_snapshot

DATA_DIR = os.environ.get("CUTSCOPE_DATA_DIR")
PRICE = Pricing(2.5, "cutscope-assumption-v1")
POLICY = IdlePolicy(retained_hours=4, point_realization=0.5)
data = load_official_data(DATA_DIR)
summary = build_summary(data.jobs, PRICE)
candidates = idle_candidates(data.jobs)
opportunity, records, analysis = build_idle_opportunity(data, PRICE, summary["total_gpu_hours"], POLICY)

## 1. Separate observed consumption from modeled reclaim
The official rule requires interactive job type, more than four hours runtime, and mean SM below 5%. The stricter scenario also requires zero mean/peak compute at job level and zero peak on every card, one attempt, valid allocation budgets, and agreement within 10%. Zero GPU compute does not imply that CPU-only interactive work is unnecessary.

In [ ]:
display(candidates.groupby(candidates.sm_util_max.eq(0)).agg(job_count=("id_job", "size"), consumed_gpu_hours=("gpu_hours", "sum")).rename_axis("zero_job_peak"))
print(f"Official candidates: {len(candidates):,}; scenario jobs: {len(records):,}; excluded: {len(analysis["exclusions"]):,}")
exclusions = pd.DataFrame(analysis["exclusions"])
display(exclusions.explode("reasons").groupby("reasons").size().rename("reason_count").to_frame())
print("A job can have multiple exclusion reasons; do not sum reason counts as jobs.")

## 2. Examine representative jobs individually
Inspect a completed zero-compute session, a cancelled zero-compute session, a session with some peak activity, and a duration outlier. These examples support investigation, not a claim that every session was abandoned. Change `JOB_ID` below to inspect another case.

In [ ]:
examples = []
for label, subset in [("completed zero peak", candidates[candidates.state_name.eq("COMPLETED") & candidates.sm_util_max.eq(0)]),
                      ("cancelled zero peak", candidates[candidates.state_name.eq("CANCELLED") & candidates.sm_util_max.eq(0)]),
                      ("some observed compute", candidates[candidates.sm_util_max.gt(0)]),
                      ("duration disagreement", candidates[(candidates.gpu_hours_ratio - 1).abs().gt(.1)])]:
    if len(subset):
        row = subset.iloc[0]
        examples.append({"example": label, **row[["id_job", "state_name", "gpu_count", "gpu_hours", "gpu_hours_alloc", "walltime_sec", "sm_util_avg", "sm_util_max", "mem_used_frac", "attempts"]].to_dict()})
display(pd.DataFrame(examples))

In [ ]:
JOB_ID = int(examples[0]["id_job"])
display(data.jobs[data.jobs.id_job.eq(JOB_ID)].T)
display(data.gpus[data.gpus.id_job.eq(JOB_ID)])
for finding in data.findings:
    if (finding.get("metadata") or {}).get("job_id") == JOB_ID:
        display(finding)

## 3. Inspect the allocation ledger
Each modeled job has one primary owner. The elapsed window after the retained hours is hypothetical; no idle onset was observed. Upper reclaim is capped to the lesser of measured and scheduler allocation hours, minus retained GPU-hours. Supporting findings are preserved but their impacts are not added.

In [ ]:
ledger = pd.DataFrame(analysis["primary_ledger"])
display(ledger.head(15))
assert ledger.job_id.is_unique
assert all(0 <= row["gpu_hours"]["low"] <= row["gpu_hours"]["point"] <= row["gpu_hours"]["high"] <= row["budget_gpu_hours"] for row in analysis["primary_ledger"])

## 4. Low / point / high
Low is zero because observed candidate evidence does not guarantee policy reclaim. Point assumes half the modeled budget is reclaimed. High assumes all of that budget is reclaimed. The 50% fraction is an explicit uncalibrated assumption, not a probability or empirical forecast. These are not statistical confidence intervals, and priced resource value is not guaranteed cash savings.

In [ ]:
scenarios = pd.DataFrame({"gpu_hours": analysis["scenarios_gpu_hours"], "resource_value_usd": analysis["scenarios_usd"]})
scenarios["sample_share_percent"] = scenarios.gpu_hours / summary["total_gpu_hours"] * 100
display(scenarios)
ax = scenarios.gpu_hours.plot.bar(figsize=(7, 4), title="Hypothetical reclaim under four-hour retained period")
ax.set_ylabel("Modeled GPU-hours, four-month sample")
plt.tight_layout()
plt.show()

## 5. Change the assumptions
Sensitivity uses the exported cohort to isolate policy parameter changes. A longer retained period reduces reclaim. The fraction represents realized budget, not an observed false-positive rate. Longer-period exports may contain fewer affected job references where the upper budget becomes zero.

In [ ]:
selected = data.jobs[data.jobs.id_job.isin([row["job_id"] for row in records])]
sensitivity = []
for retained in (4, 8, 12, 24):
    for realization in (0.25, 0.5, 0.75, 1.0):
        policy = IdlePolicy(retained, realization)
        point = sum(idle_hour_scenarios(row.gpu_hours, row.gpu_hours_alloc, row.gpu_count, policy)["point"] for row in selected.itertuples())
        sensitivity.append({"retained_hours": retained, "assumed_realization": realization, "point_gpu_hours": point, "resource_value_usd": PRICE.cost(point)})
sensitivity = pd.DataFrame(sensitivity)
display(sensitivity)
ax = sensitivity.pivot(index="retained_hours", columns="assumed_realization", values="point_gpu_hours").plot(marker="o", figsize=(8, 4), title="Sensitivity to retained period and assumed realization")
ax.set(xlabel="Retained session period (hours)", ylabel="Point modeled GPU-hours")
plt.tight_layout()
plt.show()

## 6. What it costs if we are wrong
Actual downside USD and recoverability confidence remain null. A replay sensitivity assumes a stated fraction of eligible measured GPU-hours needs one identical-cost replay. This is a consumption-weighted assumption, not an observed fraction of interrupted jobs or a business-loss bound. CPU-only work, engineer time and lost unsaved state are not priced. Disabling enforcement does not restore unsaved work.

In [ ]:
display(opportunity["cost_if_wrong"])
display(pd.DataFrame(analysis["replay_sensitivity"]))
print(analysis["replay_basis"])
print("Recommended sequence: warning-only pilot → review legitimate CPU-only work → opt-outs and checkpoints → limited enforcement with rollback criteria.")

## 7. Confirm the backend handoff
The existing API gets low/high amounts, method text, null confidence/downside amounts, all affected job references, and original finding evidence. Structured point estimates and ledger are producer metadata and are not exposed automatically by the API. The method text also explains the point assumption and result.

In [ ]:
snapshot = build_snapshot(data, PRICE, POLICY)
assert snapshot["summary"] == summary
assert snapshot["opportunities"][0] == opportunity
assert snapshot["jobs"] == records
assert opportunity["job_count"] == len(opportunity["jobs"]) == len(records)
assert {ref["job_id"] for ref in opportunity["jobs"]} == {row["job_id"] for row in records}
print(f"Ready to validate: {len(records):,} affected jobs with complete references and evidence")

## Next stage
Validate the policy with operational feedback before treating it as deployable savings. Expand GPU-not-needed, slow-cancel and imbalance analysis using this same primary ledger. Whole-job ownership prevents overlap conservatively; supporting findings remain visible. Live MantisGrid causal/tool integration belongs to the AI owner.